# RoboTwin 2.0 — SAPIEN 双臂操作数据生成 & 基准一键预览

_One-click preview of RoboTwin 2.0 — SAPIEN-based bimanual data generator & benchmark._

基于 [RoboTwin-Platform/RoboTwin](https://github.com/RoboTwin-Platform/RoboTwin)（CVPR 2025 Highlight）。RoboTwin 用 SAPIEN 跑**双臂协作操作**，自带专家规划器（curobo + mplib）批量生成示范数据，并支持强域随机化。本仓 `sapien-experience` 的 SAPIEN 系基准之一（与 [ManiSkill](./ManiSkill.ipynb) 并列）。

_Built on RoboTwin (CVPR 2025 Highlight). SAPIEN-based **bimanual** manipulation with scripted expert planners (curobo + mplib) that mass-generate demos, plus strong domain randomization. The other SAPIEN benchmark in this repo, alongside [ManiSkill](./ManiSkill.ipynb)._

## 三轴一键预览 / Three-axis one-click preview

| 轴 / Axis | 内容 / Coverage | 章节 |
|---|---|---|
| **任务 / Task** | 50 个双臂任务：交接 · 堆叠 · 摆放 · 开合 / 50 bimanual tasks: handover · stack · place · articulated | §1–§2 |
| **求解器 / Solver** | ⭐ 专家规划器现场求解（无需训练）/ expert planner solves it live, no training | §2 |
| **域随机化 / Domain rand.** | 干净 vs 杂乱桌面/随机光照背景 / clean vs cluttered + random light/bg | §3 |

**机制 / Mechanism**：`scripts/robotwin_demo.py` 复用上游 `script/collect_data.py` 的种子搜索阶段（跑专家 `play_once()`），只翻开 `render_freq>0` 弹 SAPIEN viewer。

**前置 / Prereqs**：`conda` · NVIDIA GPU（CUDA）· 显示器 `DISPLAY=:0`（预览用）· ~10+ GB 磁盘（curobo + 资产 ~GB）。

> ⚠️ RoboTwin **不发布预训练策略 ckpt**；能看到机器人动起来的是**内置专家规划器**（也是它生成 10 万条轨迹数据集的引擎）。策略训练/评测见 §4。

---
## 0. 安装 conda env `robotwin_sx`（幂等 / idempotent）

**做了什么 / What it does**（见 `scripts/install_robotwin_env.sh`）：
1. `conda create -n robotwin_sx python=3.10`
2. `pip install -r script/requirements.txt`（torch 2.4.1 · sapien 3.0.0b1 · mplib · ...）
3. `pip install pytorch3d`（源码编译）
4. 补丁 `sapien/urdf_loader.py`（utf-8）+ `mplib/planner.py`（去掉 collide 短路）
5. `clone + pip install curobo v0.7.8`（GPU 运动规划，编 CUDA 核，慢）
6. 下载并解压资产（background_texture / embodiments / objects，~GB，来自 HF `TianxingChen/RoboTwin2.0`）

> ⏱️ **首次 30–60 分钟**（curobo 编译 + 资产下载）；已装好则秒跳过。独立 env 是刚需——`sapien==3.0.0b1` / `torch==2.4.1` 的硬 pin 会和 ManiSkill 的 sapien≥3.0 冲突。

_First run 30–60 min (curobo build + asset download); re-run is seconds. Isolated env is mandatory — RoboTwin hard-pins `sapien==3.0.0b1` / `torch==2.4.1`, clashing with ManiSkill's sapien≥3.0._

In [ ]:
!bash scripts/install_robotwin_env.sh

---
## 1. 看有哪些任务 (Task List)

_50 bimanual tasks. Use any name below as the `<task>` argument in §2._

In [ ]:
!conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py list

---
## 2. ⭐ 专家规划器现场求解 (Expert Planner — Solve Live)

每个 cell 打开 SAPIEN viewer，让**专家规划器**（curobo 解 IK + mplib 规划）真去把任务做出来——双臂抓取、交接、堆叠、对位。`--episodes 2` 连放 2 个成功 episode，`Ctrl-C` 退出。按难度/类型分组。

_Each cell opens the SAPIEN viewer and lets the **expert planner** (curobo IK + mplib) actually solve the task — dual-arm grasp / handover / stack / align. `--episodes 2` plays 2 successful episodes; `Ctrl-C` to quit._

> 💡 看到的轨迹就是 RoboTwin 数据集的来源。换任意 §1 里的任务名即可。

### 2.1 交接 / Handover

In [ ]:
# 🧱 双臂交接方块 / hand a block arm-to-arm
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview handover_block --episodes 2 --gpu 0

In [ ]:
# 🎤 双臂交接话筒 / hand over a microphone
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview handover_mic --episodes 2 --gpu 0

### 2.2 堆叠 / Stacking

In [ ]:
# 🧱 叠两个方块 / stack two blocks
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview stack_blocks_two --episodes 2 --gpu 0

In [ ]:
# 🥣 叠两个碗 / stack two bowls
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview stack_bowls_two --episodes 2 --gpu 0

In [ ]:
# 📊 按大小排方块 / rank blocks by size
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview blocks_ranking_size --episodes 2 --gpu 0

### 2.3 拾取与摆放 / Pick & Place

In [ ]:
# 🍾 双臂各抓一瓶 / pick two bottles
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview pick_dual_bottles --episodes 2 --gpu 0

In [ ]:
# 🧺 物体放进篮子 / place object into basket
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview place_object_basket --episodes 2 --gpu 0

In [ ]:
# 👟 摆好一双鞋 / arrange a pair of shoes
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview place_dual_shoes --episodes 2 --gpu 0

In [ ]:
# 🍽️ 容器放到盘上 / place container on plate
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview place_container_plate --episodes 2 --gpu 0

### 2.4 工具/接触 & 铰接物体 / Tool-use & Articulated

In [ ]:
# 🔨 用锤子敲方块 / beat block with hammer
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview beat_block_hammer --episodes 2 --gpu 0

In [ ]:
# 🍲 双臂抬锅 / lift a pot with both arms
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview lift_pot --episodes 2 --gpu 0

In [ ]:
# 💻 开合笔记本 / open a laptop
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview open_laptop --episodes 2 --gpu 0

In [ ]:
# 🌡️ 开微波炉门 / open the microwave
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview open_microwave --episodes 2 --gpu 0

In [ ]:
# 🔘 拨开关 / flip a switch
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview turn_switch --episodes 2 --gpu 0

In [ ]:
# 🔔 按服务铃 / click a bell
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview click_bell --episodes 2 --gpu 0

---
## 3. 域随机化对照 (Domain Randomization)

RoboTwin 2.0 的招牌：强域随机化（杂乱桌面 + 随机背景纹理 + 随机光照 + 随机桌高/相机距离）。同一任务 `demo_clean` vs `demo_randomized` 对照看。

_RoboTwin 2.0's headline: strong domain randomization (cluttered table + random background + random light + random table height/camera distance). Same task, clean vs randomized._

### 3.1 干净 / Clean

In [ ]:
# 🧱 叠方块（干净桌面） / stack blocks, clean table
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview stack_blocks_two --episodes 2 --gpu 0

### 3.2 域随机化 / Randomized

In [ ]:
# 🧱 叠方块（杂乱+随机光照背景） / stack blocks, cluttered + random light/bg
!DISPLAY=:0 conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py preview stack_blocks_two --config demo_randomized --episodes 2 --gpu 0

---
## 4. 生成示范数据 + 策略训练/评测 (Data Generation & Policies)

### 4.1 无头批量生成数据 / Headless data collection

专家规划器无头跑、写 HDF5（RGB + depth + qpos + endpose）。落在 `dependencies/RoboTwin/data/<task>/_run_demo_clean/`。

_Expert planner runs headless and writes HDF5 (RGB + depth + qpos + endpose) under `data/<task>/...`._

In [ ]:
# 收集 stack_blocks_two 的 20 条示范 / collect 20 demos
!conda run -n robotwin_sx --no-capture-output python scripts/robotwin_demo.py collect stack_blocks_two --config demo_clean --episodes 20 --gpu 0

### 4.2 策略 baselines（需自训）/ Policy baselines (train-your-own)

RoboTwin 集成 DP · ACT · DP3 · RDT · π0 · π0.5 · OpenVLA-OFT · TinyVLA · DexVLA 等（在 `dependencies/RoboTwin/policy/`），但**不发布预训练 ckpt**——需自己收数据→训练→评测：

_RoboTwin integrates DP / ACT / DP3 / RDT / π0 / π0.5 / OpenVLA-OFT / TinyVLA / DexVLA under `policy/`, but ships **no pretrained ckpts** — collect → train → eval yourself:_

```bash
# 评测一个训好的 Diffusion Policy（示意）/ evaluate a trained DP (illustrative)
cd dependencies/RoboTwin
bash policy/DP/eval.sh stack_blocks_two demo_clean aloha-agilex 100 0 0
```

- **现成数据集**：HF [`TianxingChen/RoboTwin2.0`](https://huggingface.co/datasets/TianxingChen/RoboTwin2.0)（10 万+ 轨迹），可跳过自己收集。
- **CVPR 2025 挑战赛分支**：见 README 的 Challenge 分支链接。